In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

In [2]:


mlflow.set_experiment("uber-demand-prediction")

2026/05/27 17:20:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/27 17:20:26 INFO mlflow.store.db.utils: Updating database tables
2026/05/27 17:20:27 INFO mlflow.tracking.fluent: Experiment with name 'uber-demand-prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/Soham/Documents/notebooks/mlruns/1', creation_time=1779882627494, experiment_id='1', last_update_time=1779882627494, lifecycle_stage='active', name='uber-demand-prediction', tags={}, trace_location=None, workspace='default'>

In [3]:
import os

os.getcwd()

'c:\\Users\\Soham\\Documents\\UBER DEMAND PREDICATION'

In [39]:
# load the training and test data

train_data_path = "data/train.csv"
test_data_path = "data/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")


In [37]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [8]:
# missing values in the test data

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [9]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [10]:
X_train.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4


In [11]:
# make X_test and y_test

X_test = test_df.drop(columns=["total_pickups"])

y_test = test_df["total_pickups"]

In [12]:
X_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,36.0,44.0,31.0,29.0,0,38.0,1
2016-03-01 00:15:00,41.0,36.0,44.0,31.0,0,39.0,1
2016-03-01 00:30:00,35.0,41.0,36.0,44.0,0,37.0,1
2016-03-01 00:45:00,47.0,35.0,41.0,36.0,0,41.0,1
2016-03-01 01:00:00,34.0,47.0,35.0,41.0,0,38.0,1


In [13]:
from sklearn import set_config

set_config(transform_output="pandas")

In [14]:
# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1,force_int_remainder_cols=False)

In [13]:
encoder

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('ohe',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['region', 'day_of_week'])])

In [15]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


In [16]:
import optuna
import tqdm 

In [17]:
# set the experiment

mlflow.set_experiment("Model Selection")

2026/05/27 17:46:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/27 17:46:01 INFO mlflow.store.db.utils: Updating database tables
2026/05/27 17:46:05 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/Soham/Documents/UBER DEMAND PREDICATION/mlruns/1', creation_time=1779884165555, experiment_id='1', last_update_time=1779884165555, lifecycle_stage='active', name='Model Selection', tags={}, trace_location=None, workspace='default'>

In [38]:
# ---------------------------------------------------
# FEATURES AND TARGET
# ---------------------------------------------------

# train_df -> Jan + Feb
# test_df  -> March

X_full = train_df.drop(
    columns=["total_pickups"]
)

y_full = train_df[
    "total_pickups"
]

X_test = test_df.drop(
    columns=["total_pickups"]
)

y_test = test_df[
    "total_pickups"
]

# ---------------------------------------------------
# CONVERT CATEGORICAL COLUMNS
# ---------------------------------------------------

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ---------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ---------------------------------------------------

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

In [47]:
import mlflow
import mlflow.sklearn
import optuna
import numpy as np
import pandas as pd

from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ---------------------------------------------------
# sklearn output as pandas
# ---------------------------------------------------

set_config(transform_output="pandas")

# ---------------------------------------------------
# FEATURES AND TARGET
# ---------------------------------------------------

# train_df -> Jan + Feb
# test_df  -> March

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df[
    "total_pickups"
]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed[
    "total_pickups"
]

# ---------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ---------------------------------------------------

# preserve chronology

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

# ---------------------------------------------------
# PREPROCESSOR
# ---------------------------------------------------

encoder = ColumnTransformer(
    transformers=[
        (
            "ohe",
            OneHotEncoder(
                drop="first",
                sparse_output=False,
                handle_unknown="ignore"
            ),
            ["region", "day_of_week"]
        )
    ],
    remainder="passthrough",
    n_jobs=-1,
    force_int_remainder_cols=False
)

# ---------------------------------------------------
# MLFLOW EXPERIMENT
# ---------------------------------------------------

mlflow.set_experiment(
    "uber-demand-prediction"
)

# ---------------------------------------------------
# OPTUNA OBJECTIVE FUNCTION
# ---------------------------------------------------

def objective(trial):

    with mlflow.start_run(nested=True):

        # -------------------------------------------
        # MODEL SEARCH SPACE
        # -------------------------------------------

        model_name = trial.suggest_categorical(
            "model_name",
            ["LR", "RF", "GBR", "XGBR"]
        )

        # -------------------------------------------
        # LINEAR REGRESSION
        # -------------------------------------------

        if model_name == "LR":

            model = LinearRegression()

        # -------------------------------------------
        # RANDOM FOREST
        # -------------------------------------------

        elif model_name == "RF":

            model = RandomForestRegressor(
                n_estimators=trial.suggest_int(
                    "n_estimators_rf",
                    10,
                    100,
                    step=10
                ),
                max_depth=trial.suggest_int(
                    "max_depth_rf",
                    3,
                    10
                ),
                random_state=42,
                n_jobs=-1
            )

        # -------------------------------------------
        # GRADIENT BOOSTING
        # -------------------------------------------

        elif model_name == "GBR":

            model = GradientBoostingRegressor(
                n_estimators=trial.suggest_int(
                    "n_estimators_gb",
                    10,
                    100,
                    step=10
                ),
                learning_rate=trial.suggest_float(
                    "learning_rate_gb",
                    1e-4,
                    1e-1,
                    log=True
                ),
                random_state=42
            )

        # -------------------------------------------
        # XGBOOST
        # -------------------------------------------

        else:

            model = XGBRegressor(
                n_estimators=trial.suggest_int(
                    "n_estimators_xgb",
                    10,
                    100,
                    step=10
                ),
                learning_rate=trial.suggest_float(
                    "learning_rate_xgb",
                    1e-4,
                    1e-1,
                    log=True
                ),
                max_depth=trial.suggest_int(
                    "max_depth_xgb",
                    3,
                    10
                ),
                random_state=42,
                n_jobs=-1
            )

        # -------------------------------------------
        # PIPELINE
        # -------------------------------------------

        pipe = Pipeline([
            ("encoder", encoder),
            ("model", model)
        ])

        # -------------------------------------------
        # TRAIN
        # -------------------------------------------

        pipe.fit(
            X_train,
            y_train
        )

        # -------------------------------------------
        # VALIDATION PREDICTIONS
        # -------------------------------------------

        y_pred = pipe.predict(
            X_val
        )

        # -------------------------------------------
        # METRICS
        # -------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # -------------------------------------------
        # LOGGING
        # -------------------------------------------

        mlflow.log_param(
            "model_name",
            model_name
        )

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2_SCORE",
            r2
        )

        return mape

# ---------------------------------------------------
# PARENT RUN
# ---------------------------------------------------

with mlflow.start_run(
    run_name="hyperparameter_tuning"
):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(
        objective,
        n_trials=50
    )

    # -----------------------------------------------
    # BEST PARAMETERS
    # -----------------------------------------------

    best_params = study.best_params

    print("Best Params:")
    print(best_params)

    print("\nBest MAPE:")
    print(study.best_value)

# ---------------------------------------------------
# TRAIN FINAL MODEL ON TRAIN + VALIDATION
# ---------------------------------------------------

best_model_name = best_params[
    "model_name"
]

if best_model_name == "LR":

    final_model = LinearRegression()

elif best_model_name == "RF":

    final_model = RandomForestRegressor(
        n_estimators=best_params[
            "n_estimators_rf"
        ],
        max_depth=best_params[
            "max_depth_rf"
        ],
        random_state=42,
        n_jobs=-1
    )

elif best_model_name == "GBR":

    final_model = GradientBoostingRegressor(
        n_estimators=best_params[
            "n_estimators_gb"
        ],
        learning_rate=best_params[
            "learning_rate_gb"
        ],
        random_state=42
    )

else:

    final_model = XGBRegressor(
        n_estimators=best_params[
            "n_estimators_xgb"
        ],
        learning_rate=best_params[
            "learning_rate_xgb"
        ],
        max_depth=best_params[
            "max_depth_xgb"
        ],
        random_state=42,
        n_jobs=-1
    )

# ---------------------------------------------------
# COMBINE TRAIN + VALIDATION
# ---------------------------------------------------

X_final_train = pd.concat([
    X_train,
    X_val
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ---------------------------------------------------
# FINAL PIPELINE
# ---------------------------------------------------

final_pipe = Pipeline([
    ("encoder", encoder),
    ("model", final_model)
])

# ---------------------------------------------------
# TRAIN FINAL MODEL
# ---------------------------------------------------

final_pipe.fit(
    X_final_train,
    y_final_train
)

# ---------------------------------------------------
# FINAL TEST EVALUATION
# ---------------------------------------------------

y_test_pred = final_pipe.predict(
    X_test
)

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ---------------------------------------------------
# LOG FINAL MODEL
# ---------------------------------------------------

with mlflow.start_run(
    run_name="best_final_model"
):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.sklearn.log_model(
        final_pipe,
        artifact_path="model"
    )

print("\nFinal model logged successfully.")

[I 2026-05-28 00:39:19,126] A new study created in memory with name: no-name-cde875ac-b2dc-47fc-88ef-886d2dee4d50
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
[I 2026-05-28 00:39:25,467] Trial 0 finished with value: 14.17186450958252 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 10, 'learning_rate_xgb': 0.005302723094617279, 'max_depth_xgb': 8}. Best is trial 0 with value: 14.17186450958252.
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
[I 2026-05-28 00:39:54,156] Trial 1 finished with value: 14.221196558

Best Params:
{'model_name': 'RF', 'n_estimators_rf': 80, 'max_depth_rf': 8}

Best MAPE:
0.4208446473528587


2026/05/28 00:47:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 00:47:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



FINAL TEST RESULTS
MAPE: 0.2976028832369568
MAE: 14.107183947394265
RMSE: 22.588830928607617
R2: 0.9700961725994214

Final model logged successfully.


In [64]:
# ---------------------------------------------------
# CREATE TIME FEATURES FOR TRAIN DATA
# ---------------------------------------------------

df = train_df.copy()

# make sure datetime index exists
df.index = pd.to_datetime(df.index)

# -----------------------------------------------
# BASIC TIME FEATURES
# -----------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# -----------------------------------------------
# CYCLICAL ENCODING
# -----------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# -----------------------------------------------
# LAG FEATURES
# -----------------------------------------------

# previous hour pickups
df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

# previous 24 hour pickups
df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

# previous 168 hours = previous week
df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# -----------------------------------------------
# ROLLING FEATURES
# -----------------------------------------------

# rolling average over 24 hours
df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

# rolling std over 24 hours
df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

# -----------------------------------------------
# DROP NULLS CREATED BY SHIFTS
# -----------------------------------------------

df = df.dropna()

# ---------------------------------------------------
# CREATE SAME FEATURES FOR TEST DATA
# ---------------------------------------------------

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# -----------------------------------------------
# BASIC TIME FEATURES
# -----------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# -----------------------------------------------
# CYCLICAL ENCODING
# -----------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# -----------------------------------------------
# LAG FEATURES
# -----------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# -----------------------------------------------
# ROLLING FEATURES
# -----------------------------------------------

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

# -----------------------------------------------
# DROP NULLS
# -----------------------------------------------

test_processed = test_processed.dropna()

# ---------------------------------------------------
# FEATURES AND TARGET
# ---------------------------------------------------

# train_df -> Jan + Feb
# test_df  -> March

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df[
    "total_pickups"
]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed[
    "total_pickups"
]

# ---------------------------------------------------
# CONVERT CATEGORICAL COLUMNS
# ---------------------------------------------------

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ---------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ---------------------------------------------------

# preserve chronology

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]



In [63]:
import mlflow
import mlflow.xgboost
import optuna
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ---------------------------------------------------
# FEATURES AND TARGET
# ---------------------------------------------------

# train_df -> Jan + Feb
# test_df  -> March

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df[
    "total_pickups"
]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed[
    "total_pickups"
]

# ---------------------------------------------------
# CONVERT CATEGORICAL COLUMNS
# ---------------------------------------------------

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ---------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ---------------------------------------------------

# preserve chronology

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

# ---------------------------------------------------
# MLFLOW EXPERIMENT
# ---------------------------------------------------

mlflow.set_experiment(
    "xgboost-lag-features"
)

# ---------------------------------------------------
# OBJECTIVE FUNCTION
# ---------------------------------------------------

def objective(trial):

    with mlflow.start_run(nested=True):

        # -------------------------------------------
        # MODEL
        # -------------------------------------------

        model = XGBRegressor(

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                500,
                step=50
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-3,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                12
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

            min_child_weight=trial.suggest_int(
                "min_child_weight",
                1,
                10
            ),

            gamma=trial.suggest_float(
                "gamma",
                0,
                5
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                1e-5,
                10,
                log=True
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                1e-5,
                10,
                log=True
            ),

            enable_categorical=True,

            tree_method="hist",

            random_state=42,

            n_jobs=-1
        )

        # -------------------------------------------
        # TRAIN
        # -------------------------------------------

        model.fit(
            X_train,
            y_train
        )

        # -------------------------------------------
        # VALIDATION PREDICTIONS
        # -------------------------------------------

        y_pred = model.predict(
            X_val
        )

        # -------------------------------------------
        # METRICS
        # -------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # -------------------------------------------
        # LOGGING
        # -------------------------------------------

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return mape

# ---------------------------------------------------
# OPTUNA TUNING
# ---------------------------------------------------

with mlflow.start_run(
    run_name="xgboost_lag_feature_tuning"
):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(
        objective,
        n_trials=50,
        n_jobs=1
    )

    # log best params
    mlflow.log_params(
        study.best_params
    )

    # log best score
    mlflow.log_metric(
        "best_validation_MAPE",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest Validation MAPE:")
print(study.best_value)

# ---------------------------------------------------
# FINAL MODEL TRAINING
# ---------------------------------------------------

best_params = study.best_params

final_model = XGBRegressor(

    n_estimators=best_params[
        "n_estimators"
    ],

    learning_rate=best_params[
        "learning_rate"
    ],

    max_depth=best_params[
        "max_depth"
    ],

    subsample=best_params[
        "subsample"
    ],

    colsample_bytree=best_params[
        "colsample_bytree"
    ],

    min_child_weight=best_params[
        "min_child_weight"
    ],

    gamma=best_params[
        "gamma"
    ],

    reg_alpha=best_params[
        "reg_alpha"
    ],

    reg_lambda=best_params[
        "reg_lambda"
    ],

    enable_categorical=True,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

# ---------------------------------------------------
# COMBINE TRAIN + VALIDATION
# ---------------------------------------------------

X_final_train = pd.concat([
    X_train,
    X_val
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ---------------------------------------------------
# TRAIN FINAL MODEL
# ---------------------------------------------------

final_model.fit(
    X_final_train,
    y_final_train
)

# ---------------------------------------------------
# TEST EVALUATION
# ---------------------------------------------------

y_test_pred = final_model.predict(
    X_test
)

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ---------------------------------------------------
# LOG FINAL MODEL
# ---------------------------------------------------

with mlflow.start_run(
    run_name="best_xgboost_lag_model"
):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.xgboost.log_model(
        xgb_model=final_model,
        name="model"
    )

print("\nFinal model logged successfully.")

[I 2026-05-28 03:24:31,690] A new study created in memory with name: no-name-528fde74-0441-4e5d-a4fc-295ece127cc3
[I 2026-05-28 03:24:34,602] Trial 0 finished with value: 2.545433759689331 and parameters: {'n_estimators': 250, 'learning_rate': 0.007681028760174494, 'max_depth': 6, 'subsample': 0.6607349104529298, 'colsample_bytree': 0.771704062043337, 'min_child_weight': 7, 'gamma': 2.072211533575179, 'reg_alpha': 3.217992447946572, 'reg_lambda': 1.4176417371919321e-05}. Best is trial 0 with value: 2.545433759689331.
[I 2026-05-28 03:24:35,780] Trial 1 finished with value: 0.9315700531005859 and parameters: {'n_estimators': 150, 'learning_rate': 0.07146363076132382, 'max_depth': 4, 'subsample': 0.8599189024843719, 'colsample_bytree': 0.839504727247235, 'min_child_weight': 10, 'gamma': 2.448249549298805, 'reg_alpha': 1.569548129962414e-05, 'reg_lambda': 0.0001253875585924831}. Best is trial 1 with value: 0.9315700531005859.
[I 2026-05-28 03:24:40,346] Trial 2 finished with value: 1.2722

Best Params:
{'n_estimators': 150, 'learning_rate': 0.04329102718899746, 'max_depth': 5, 'subsample': 0.9711168278686546, 'colsample_bytree': 0.9616174161845148, 'min_child_weight': 4, 'gamma': 4.017741318210166, 'reg_alpha': 0.025525317765738183, 'reg_lambda': 0.08130251687978225}

Best Validation MAPE:
0.6656621098518372

FINAL TEST RESULTS
MAPE: 0.2995116114616394
MAE: 13.094730377197266
RMSE: 21.209459835729845
R2: 0.9736368060112

Final model logged successfully.


## optimised rolling mean


In [65]:
import mlflow
import mlflow.xgboost
import optuna
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ===================================================
# CREATE TIME FEATURES FOR TRAIN DATA
# ===================================================

df = train_df.copy()

# ensure datetime index
df.index = pd.to_datetime(df.index)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

df["rolling_mean_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

df["rolling_std_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# CREATE SAME FEATURES FOR TEST DATA
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

test_processed["rolling_mean_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

test_processed["rolling_std_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = test_processed.dropna()

# ===================================================
# FEATURES AND TARGET
# ===================================================

# Jan + Feb
X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

# March
X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed["total_pickups"]

# ===================================================
# CATEGORICAL FEATURES
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ===================================================
# TRAIN VALIDATION SPLIT
# ===================================================

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

# ===================================================
# MLFLOW EXPERIMENT
# ===================================================

mlflow.set_experiment(
    "xgboost_rolling_features_optimized"
)

# ===================================================
# OBJECTIVE FUNCTION
# ===================================================

def objective(trial):

    with mlflow.start_run(nested=True):

        model = XGBRegressor(

            # ---------------------------------------
            # IMPROVED SEARCH SPACE
            # ---------------------------------------

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                400,
                step=50
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-2,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                8
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            min_child_weight=trial.suggest_int(
                "min_child_weight",
                2,
                10
            ),

            gamma=trial.suggest_float(
                "gamma",
                0,
                3
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                1e-4,
                5,
                log=True
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                1e-4,
                5,
                log=True
            ),

            # ---------------------------------------
            # FIXED SETTINGS
            # ---------------------------------------

            eval_metric="mae",

            enable_categorical=True,

            tree_method="hist",

            random_state=42,

            n_jobs=-1
        )

        # -------------------------------------------
        # TRAIN
        # -------------------------------------------

        model.fit(

            X_train,
            y_train,

            eval_set=[(X_val, y_val)],

            verbose=False
        )

        # -------------------------------------------
        # VALIDATION PREDICTIONS
        # -------------------------------------------

        y_pred = model.predict(X_val)

        # -------------------------------------------
        # METRICS
        # -------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # -------------------------------------------
        # LOGGING
        # -------------------------------------------

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return mape

# ===================================================
# OPTUNA TUNING
# ===================================================

with mlflow.start_run(

    run_name="xgboost_rolling_tuning"

):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(

        objective,

        n_trials=50,

        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_validation_MAPE",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest Validation MAPE:")
print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

best_params = study.best_params

final_model = XGBRegressor(

    n_estimators=best_params[
        "n_estimators"
    ],

    learning_rate=best_params[
        "learning_rate"
    ],

    max_depth=best_params[
        "max_depth"
    ],

    subsample=best_params[
        "subsample"
    ],

    colsample_bytree=best_params[
        "colsample_bytree"
    ],

    min_child_weight=best_params[
        "min_child_weight"
    ],

    gamma=best_params[
        "gamma"
    ],

    reg_alpha=best_params[
        "reg_alpha"
    ],

    reg_lambda=best_params[
        "reg_lambda"
    ],

    eval_metric="mae",

    enable_categorical=True,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

# ===================================================
# COMBINE TRAIN + VALIDATION
# ===================================================

X_final_train = pd.concat([
    X_train,
    X_val
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_final_train,
    y_final_train,

    verbose=False
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name="best_xgboost_rolling_model"

):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.xgboost.log_model(
        xgb_model=final_model,
        name="model"
    )

print("\nFinal rolling feature model logged successfully.")

2026/05/28 03:39:22 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_rolling_features_optimized' does not exist. Creating a new experiment.
[I 2026-05-28 03:39:22,330] A new study created in memory with name: no-name-6d70cd05-64db-441b-9504-bdbe274b5e8b
[I 2026-05-28 03:39:25,475] Trial 0 finished with value: 1.0074721574783325 and parameters: {'n_estimators': 200, 'learning_rate': 0.01719026705063768, 'max_depth': 8, 'subsample': 0.9018858130889311, 'colsample_bytree': 0.7229796266723448, 'min_child_weight': 10, 'gamma': 2.4191934839364264, 'reg_alpha': 0.22052877009944769, 'reg_lambda': 0.00728525435443351}. Best is trial 0 with value: 1.0074721574783325.
[I 2026-05-28 03:39:28,283] Trial 1 finished with value: 1.2757853269577026 and parameters: {'n_estimators': 300, 'learning_rate': 0.08725339321668595, 'max_depth': 4, 'subsample': 0.7799004204199196, 'colsample_bytree': 0.7964042764832425, 'min_child_weight': 3, 'gamma': 1.04568198687058, 'reg_alpha': 0.001432149582538147

Best Params:
{'n_estimators': 150, 'learning_rate': 0.04404135760706516, 'max_depth': 7, 'subsample': 0.7608960922055488, 'colsample_bytree': 0.8290579050814723, 'min_child_weight': 7, 'gamma': 1.914371013353322, 'reg_alpha': 0.00011710301793926702, 'reg_lambda': 0.1514028056166948}

Best Validation MAPE:
0.6843069195747375

FINAL TEST RESULTS
MAPE: 0.28109055757522583
MAE: 12.559831619262695
RMSE: 20.41254509046612
R2: 0.9755806922912598

Final rolling feature model logged successfully.


In [67]:
# ---------------------------------------------------
# CREATE TIME FEATURES FOR TRAIN DATA
# ---------------------------------------------------

df = train_df.copy()

# ensure datetime index
df.index = pd.to_datetime(df.index)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# EWMA FEATURES
# ---------------------------------------------------

df["ewma_24"] = (
    df["total_pickups"]
    .shift(1)
    .ewm(
        span=24,
        adjust=False
    )
    .mean()
)

df["ewma_168"] = (
    df["total_pickups"]
    .shift(1)
    .ewm(
        span=168,
        adjust=False
    )
    .mean()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# CREATE SAME FEATURES FOR TEST DATA
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# EWMA FEATURES
# ---------------------------------------------------

test_processed["ewma_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .ewm(
        span=24,
        adjust=False
    )
    .mean()
)

test_processed["ewma_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .ewm(
        span=168,
        adjust=False
    )
    .mean()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = test_processed.dropna()

# ---------------------------------------------------
# FEATURES / TARGET
# ---------------------------------------------------

# Jan + Feb
X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

# March
X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed["total_pickups"]

# ---------------------------------------------------
# CATEGORICAL COLUMNS
# ---------------------------------------------------

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ---------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ---------------------------------------------------

# preserve chronology

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

In [61]:
import mlflow
import mlflow.xgboost
import optuna
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ---------------------------------------------------
# MLFLOW EXPERIMENT
# ---------------------------------------------------

mlflow.set_experiment(
    "xgboost_optimized_forecasting"
)

# ---------------------------------------------------
# OBJECTIVE FUNCTION
# ---------------------------------------------------

def objective(trial):

    with mlflow.start_run(nested=True):

        # ---------------------------------------------------
        # MODEL
        # ---------------------------------------------------

        model = XGBRegressor(

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                400,
                step=50
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-2,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                8
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

            min_child_weight=trial.suggest_int(
                "min_child_weight",
                1,
                10
            ),

            gamma=trial.suggest_float(
                "gamma",
                0,
                5
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                1e-5,
                10,
                log=True
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                1e-5,
                10,
                log=True
            ),

            eval_metric="mae",

            enable_categorical=True,

            tree_method="hist",

            random_state=42,

            n_jobs=-1
        )

        # ---------------------------------------------------
        # TRAIN
        # ---------------------------------------------------

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        # ---------------------------------------------------
        # VALIDATION PREDICTIONS
        # ---------------------------------------------------

        y_pred = model.predict(X_val)

        # ---------------------------------------------------
        # METRICS
        # ---------------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # ---------------------------------------------------
        # LOGGING
        # ---------------------------------------------------

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return mape

# ---------------------------------------------------
# OPTUNA TUNING
# ---------------------------------------------------

with mlflow.start_run(
    run_name="xgboost_optimized_tuning"
):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(
        objective,
        n_trials=50,
        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_validation_MAPE",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest Validation MAPE:")
print(study.best_value)

# ---------------------------------------------------
# FINAL MODEL
# ---------------------------------------------------

best_params = study.best_params

final_model = XGBRegressor(

    n_estimators=best_params["n_estimators"],

    learning_rate=best_params["learning_rate"],

    max_depth=best_params["max_depth"],

    subsample=best_params["subsample"],

    colsample_bytree=best_params["colsample_bytree"],

    min_child_weight=best_params["min_child_weight"],

    gamma=best_params["gamma"],

    reg_alpha=best_params["reg_alpha"],

    reg_lambda=best_params["reg_lambda"],

    eval_metric="mae",

    enable_categorical=True,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

# ---------------------------------------------------
# COMBINE TRAIN + VALIDATION
# ---------------------------------------------------

X_final_train = pd.concat([
    X_train,
    X_val
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ---------------------------------------------------
# TRAIN FINAL MODEL
# ---------------------------------------------------

final_model.fit(
    X_final_train,
    y_final_train,
    verbose=False
)

# ---------------------------------------------------
# TEST PREDICTIONS
# ---------------------------------------------------

y_test_pred = final_model.predict(
    X_test
)

# ---------------------------------------------------
# FINAL METRICS
# ---------------------------------------------------

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ---------------------------------------------------
# LOG FINAL MODEL
# ---------------------------------------------------

with mlflow.start_run(
    run_name="best_xgboost_optimized_model"
):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.xgboost.log_model(
        xgb_model=final_model,
        name="model"
    )

print("\nFinal optimized model logged successfully.")

[I 2026-05-28 01:23:42,761] A new study created in memory with name: no-name-75a93069-10bd-4148-ac5d-5546016811e5
[I 2026-05-28 01:23:47,752] Trial 0 finished with value: 4.35491943359375 and parameters: {'n_estimators': 100, 'learning_rate': 0.013083782627749847, 'max_depth': 4, 'subsample': 0.7743296247881335, 'colsample_bytree': 0.8577124342983431, 'min_child_weight': 3, 'gamma': 4.482311652943406, 'reg_alpha': 7.95473608350255, 'reg_lambda': 5.082656779792713}. Best is trial 0 with value: 4.35491943359375.
[I 2026-05-28 01:23:52,189] Trial 1 finished with value: 4.153049945831299 and parameters: {'n_estimators': 100, 'learning_rate': 0.014424281904637477, 'max_depth': 3, 'subsample': 0.9745304636777429, 'colsample_bytree': 0.9894916188145957, 'min_child_weight': 1, 'gamma': 0.3626856844217269, 'reg_alpha': 0.011668713483764345, 'reg_lambda': 0.8512529691356878}. Best is trial 1 with value: 4.153049945831299.
[I 2026-05-28 01:24:06,989] Trial 2 finished with value: 0.958111524581909

Best Params:
{'n_estimators': 250, 'learning_rate': 0.02594581596138249, 'max_depth': 5, 'subsample': 0.8835169503514262, 'colsample_bytree': 0.8909488709757085, 'min_child_weight': 5, 'gamma': 0.45194492253057383, 'reg_alpha': 0.0018002505243964934, 'reg_lambda': 0.005642524404423381}

Best Validation MAPE:
0.6397348642349243

FINAL TEST RESULTS
MAPE: 0.2983592748641968
MAE: 13.123501777648926
RMSE: 21.285169512177003
R2: 0.9734482169151306

Final optimized model logged successfully.


In [68]:
import mlflow
import mlflow.xgboost
import optuna
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ===================================================
# CREATE TIME FEATURES FOR TRAIN DATA
# ===================================================

df = train_df.copy()

# ensure datetime index
df.index = pd.to_datetime(df.index)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# EWMA FEATURES
# ---------------------------------------------------

df["ewma_24"] = (
    df["total_pickups"]
    .shift(1)
    .ewm(
        span=24,
        adjust=False
    )
    .mean()
)

df["ewma_168"] = (
    df["total_pickups"]
    .shift(1)
    .ewm(
        span=168,
        adjust=False
    )
    .mean()
)

# ---------------------------------------------------
# OPTIONAL VOLATILITY FEATURE
# ---------------------------------------------------

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# CREATE SAME FEATURES FOR TEST DATA
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# EWMA FEATURES
# ---------------------------------------------------

test_processed["ewma_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .ewm(
        span=24,
        adjust=False
    )
    .mean()
)

test_processed["ewma_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .ewm(
        span=168,
        adjust=False
    )
    .mean()
)

# ---------------------------------------------------
# OPTIONAL VOLATILITY FEATURE
# ---------------------------------------------------

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = test_processed.dropna()

# ===================================================
# FEATURES AND TARGET
# ===================================================

# Jan + Feb
X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

# March
X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed["total_pickups"]

# ===================================================
# CATEGORICAL FEATURES
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ===================================================
# TRAIN VALIDATION SPLIT
# ===================================================

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

# ===================================================
# MLFLOW EXPERIMENT
# ===================================================

mlflow.set_experiment(
    "xgboost_ewma_features_optimized"
)

# ===================================================
# OBJECTIVE FUNCTION
# ===================================================

def objective(trial):

    with mlflow.start_run(nested=True):

        model = XGBRegressor(

            # ---------------------------------------
            # IMPROVED SEARCH SPACE
            # ---------------------------------------

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                400,
                step=50
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-2,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                8
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            min_child_weight=trial.suggest_int(
                "min_child_weight",
                2,
                10
            ),

            gamma=trial.suggest_float(
                "gamma",
                0,
                3
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                1e-4,
                5,
                log=True
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                1e-4,
                5,
                log=True
            ),

            # ---------------------------------------
            # FIXED SETTINGS
            # ---------------------------------------

            eval_metric="mae",

            enable_categorical=True,

            tree_method="hist",

            random_state=42,

            n_jobs=-1
        )

        # -------------------------------------------
        # TRAIN
        # -------------------------------------------

        model.fit(

            X_train,
            y_train,

            eval_set=[(X_val, y_val)],

            verbose=False
        )

        # -------------------------------------------
        # VALIDATION PREDICTIONS
        # -------------------------------------------

        y_pred = model.predict(X_val)

        # -------------------------------------------
        # METRICS
        # -------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # -------------------------------------------
        # LOGGING
        # -------------------------------------------

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return mape

# ===================================================
# OPTUNA TUNING
# ===================================================

with mlflow.start_run(

    run_name="xgboost_ewma_tuning"

):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(

        objective,

        n_trials=50,

        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_validation_MAPE",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest Validation MAPE:")
print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

best_params = study.best_params

final_model = XGBRegressor(

    n_estimators=best_params[
        "n_estimators"
    ],

    learning_rate=best_params[
        "learning_rate"
    ],

    max_depth=best_params[
        "max_depth"
    ],

    subsample=best_params[
        "subsample"
    ],

    colsample_bytree=best_params[
        "colsample_bytree"
    ],

    min_child_weight=best_params[
        "min_child_weight"
    ],

    gamma=best_params[
        "gamma"
    ],

    reg_alpha=best_params[
        "reg_alpha"
    ],

    reg_lambda=best_params[
        "reg_lambda"
    ],

    eval_metric="mae",

    enable_categorical=True,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

# ===================================================
# COMBINE TRAIN + VALIDATION
# ===================================================

X_final_train = pd.concat([
    X_train,
    X_val
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_final_train,
    y_final_train,

    verbose=False
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name="best_xgboost_ewma_model"

):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.xgboost.log_model(
        xgb_model=final_model,
        name="model"
    )

print("\nFinal EWMA feature model logged successfully.")

2026/05/28 03:46:14 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_ewma_features_optimized' does not exist. Creating a new experiment.
[I 2026-05-28 03:46:14,537] A new study created in memory with name: no-name-f5928189-059a-439d-ba7a-b29eeb217c8f
[I 2026-05-28 03:46:16,143] Trial 0 finished with value: 0.789021909236908 and parameters: {'n_estimators': 150, 'learning_rate': 0.060551415920620695, 'max_depth': 6, 'subsample': 0.8475949817304327, 'colsample_bytree': 0.9450014515010138, 'min_child_weight': 6, 'gamma': 1.0959500898014203, 'reg_alpha': 2.5495139712536865, 'reg_lambda': 0.024023000519088}. Best is trial 0 with value: 0.789021909236908.
[I 2026-05-28 03:46:22,000] Trial 1 finished with value: 0.8823959231376648 and parameters: {'n_estimators': 400, 'learning_rate': 0.0271826737013035, 'max_depth': 8, 'subsample': 0.7426470115045942, 'colsample_bytree': 0.9651703881669036, 'min_child_weight': 9, 'gamma': 2.386687603433708, 'reg_alpha': 3.6124185792173273, 'reg_lam

Best Params:
{'n_estimators': 300, 'learning_rate': 0.018686532699397313, 'max_depth': 5, 'subsample': 0.8062000235765406, 'colsample_bytree': 0.8324325476370543, 'min_child_weight': 6, 'gamma': 1.4234779184446682, 'reg_alpha': 0.0007981210239756229, 'reg_lambda': 0.014520919433377305}

Best Validation MAPE:
0.6411528587341309

FINAL TEST RESULTS
MAPE: 0.3091476857662201
MAE: 13.235971450805664
RMSE: 21.398990129547073
R2: 0.9731634855270386

Final EWMA feature model logged successfully.


In [33]:
import mlflow
import mlflow.lightgbm
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ---------------------------------------------------
# COPY DATA
# ---------------------------------------------------

X_train_lgb = X_train.copy()

X_val_lgb = X_val.copy()

X_test_lgb = X_test.copy()

# ---------------------------------------------------
# CATEGORY ENCODING
# ---------------------------------------------------

for col in categorical_cols:

    X_train_lgb[col] = (
        X_train_lgb[col]
        .cat.codes
    )

    X_val_lgb[col] = (
        X_val_lgb[col]
        .cat.codes
    )

    X_test_lgb[col] = (
        X_test_lgb[col]
        .cat.codes
    )

# ---------------------------------------------------
# MLFLOW EXPERIMENT
# ---------------------------------------------------

mlflow.set_experiment(
    "lightgbm_ewma_forecasting"
)

# ---------------------------------------------------
# OBJECTIVE FUNCTION
# ---------------------------------------------------

def objective(trial):

    with mlflow.start_run(nested=True):

        model = LGBMRegressor(

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                500,
                step=50
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-2,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                10
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

            random_state=42,

            n_jobs=-1
        )

        # ---------------------------------------------------
        # TRAIN
        # ---------------------------------------------------

        model.fit(
            X_train_lgb,
            y_train
        )

        # ---------------------------------------------------
        # VALIDATION PREDICTIONS
        # ---------------------------------------------------

        y_pred = model.predict(
            X_val_lgb
        )

        # ---------------------------------------------------
        # METRICS
        # ---------------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # ---------------------------------------------------
        # LOGGING
        # ---------------------------------------------------

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return mape

# ---------------------------------------------------
# OPTUNA SEARCH
# ---------------------------------------------------

with mlflow.start_run(
    run_name="lightgbm_tuning"
):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(
        objective,
        n_trials=50,
        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_validation_MAPE",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest Validation MAPE:")
print(study.best_value)

# ---------------------------------------------------
# FINAL MODEL
# ---------------------------------------------------

best_params = study.best_params

final_model = LGBMRegressor(

    **best_params,

    random_state=42,

    n_jobs=-1
)

# combine train + validation

X_final_train = pd.concat([
    X_train_lgb,
    X_val_lgb
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ---------------------------------------------------
# TRAIN FINAL MODEL
# ---------------------------------------------------

final_model.fit(
    X_final_train,
    y_final_train
)

# ---------------------------------------------------
# TEST PREDICTIONS
# ---------------------------------------------------

y_test_pred = final_model.predict(
    X_test_lgb
)

# ---------------------------------------------------
# FINAL METRICS
# ---------------------------------------------------

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ---------------------------------------------------
# LOG FINAL MODEL
# ---------------------------------------------------

with mlflow.start_run(
    run_name="best_lightgbm_model"
):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.lightgbm.log_model(
        lgb_model=final_model,
        name="model"
    )

print("\nFinal LightGBM model logged successfully.")

[I 2026-05-27 19:55:06,515] A new study created in memory with name: no-name-21f2291e-e84c-4bab-acdf-8db45141504b


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012782 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2386
[LightGBM] [Info] Number of data points in the train set: 103506, number of used features: 15
[LightGBM] [Info] Start training from score 123.678386


[I 2026-05-27 19:55:12,232] Trial 0 finished with value: 0.1988987317413726 and parameters: {'n_estimators': 150, 'learning_rate': 0.02117232841508861, 'max_depth': 9, 'subsample': 0.7034271793111397, 'colsample_bytree': 0.8406804307955399}. Best is trial 0 with value: 0.1988987317413726.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009611 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2386
[LightGBM] [Info] Number of data points in the train set: 103506, number of used features: 15
[LightGBM] [Info] Start training from score 123.678386


[I 2026-05-27 19:55:17,752] Trial 1 finished with value: 0.17377484335231924 and parameters: {'n_estimators': 200, 'learning_rate': 0.019605503299010143, 'max_depth': 10, 'subsample': 0.9527757602263098, 'colsample_bytree': 0.8070387624931069}. Best is trial 1 with value: 0.17377484335231924.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062240 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2386
[LightGBM] [Info] Number of data points in the train set: 103506, number of used features: 15
[LightGBM] [Info] Start training from score 123.678386


[I 2026-05-27 20:04:14,778] Trial 49 finished with value: 0.16285002758188818 and parameters: {'n_estimators': 400, 'learning_rate': 0.012366224607050842, 'max_depth': 8, 'subsample': 0.90480080457526, 'colsample_bytree': 0.8626721601744559}. Best is trial 22 with value: 0.15847122047172268.


Best Params:
{'n_estimators': 300, 'learning_rate': 0.023562137211794183, 'max_depth': 9, 'subsample': 0.651678649689026, 'colsample_bytree': 0.8664093472671426}

Best Validation MAPE:
0.15847122047172268
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005254 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2392
[LightGBM] [Info] Number of data points in the train set: 138009, number of used features: 15
[LightGBM] [Info] Start training from score 138.705606

FINAL TEST RESULTS
MAPE: 0.4386666294510401
MAE: 8.27162367250573
RMSE: 13.017638527834626
R2: 0.9742729387614647


2026/05/27 20:04:24 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Final LightGBM model logged successfully.


In [70]:
import mlflow
import mlflow.lightgbm
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ===================================================
# CREATE TIME FEATURES FOR TRAIN DATA
# ===================================================

df = train_df.copy()

# ensure datetime index
df.index = pd.to_datetime(df.index)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

df["rolling_mean_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

df["rolling_std_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# CREATE SAME FEATURES FOR TEST DATA
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

test_processed["rolling_mean_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .mean()
)

test_processed["rolling_std_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=168)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = test_processed.dropna()

# ===================================================
# FEATURES AND TARGET
# ===================================================

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed["total_pickups"]

# ===================================================
# CATEGORICAL FEATURES
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ===================================================
# TRAIN VALIDATION SPLIT
# ===================================================

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

# ===================================================
# LIGHTGBM NEEDS NUMERIC CATEGORIES
# ===================================================

X_train_lgb = X_train.copy()

X_val_lgb = X_val.copy()

X_test_lgb = X_test.copy()

for col in categorical_cols:

    X_train_lgb[col] = (
        X_train_lgb[col]
        .cat.codes
    )

    X_val_lgb[col] = (
        X_val_lgb[col]
        .cat.codes
    )

    X_test_lgb[col] = (
        X_test_lgb[col]
        .cat.codes
    )

# ===================================================
# MLFLOW EXPERIMENT
# ===================================================

mlflow.set_experiment(
    "lightgbm_rolling_optimized"
)

# ===================================================
# OBJECTIVE FUNCTION
# ===================================================

def objective(trial):

    with mlflow.start_run(nested=True):

        model = LGBMRegressor(

            # ---------------------------------------
            # OPTIMIZED SEARCH SPACE
            # ---------------------------------------

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                500,
                step=50
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-2,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                8
            ),

            num_leaves=trial.suggest_int(
                "num_leaves",
                20,
                150,
                step=10
            ),

            min_child_samples=trial.suggest_int(
                "min_child_samples",
                10,
                50,
                step=5
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                1e-4,
                5,
                log=True
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                1e-4,
                5,
                log=True
            ),

            objective="regression",

            random_state=42,

            n_jobs=-1
        )

        # ---------------------------------------------------
        # TRAIN
        # ---------------------------------------------------

        model.fit(

            X_train_lgb,
            y_train,

            eval_set=[(X_val_lgb, y_val)],

            eval_metric="mae"
        )

        # ---------------------------------------------------
        # VALIDATION PREDICTIONS
        # ---------------------------------------------------

        y_pred = model.predict(
            X_val_lgb
        )

        # ---------------------------------------------------
        # METRICS
        # ---------------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # ---------------------------------------------------
        # LOGGING
        # ---------------------------------------------------

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return mape

# ===================================================
# OPTUNA TUNING
# ===================================================

with mlflow.start_run(

    run_name="lightgbm_rolling_tuning"

):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(

        objective,

        n_trials=50,

        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_validation_MAPE",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest Validation MAPE:")
print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

best_params = study.best_params

final_model = LGBMRegressor(

    **best_params,

    objective="regression",

    random_state=42,

    n_jobs=-1
)

# ===================================================
# COMBINE TRAIN + VALIDATION
# ===================================================

X_final_train = pd.concat([
    X_train_lgb,
    X_val_lgb
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_final_train,
    y_final_train
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test_lgb
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name="best_lightgbm_rolling_model"

):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.lightgbm.log_model(
        lgb_model=final_model,
        name="model"
    )

print("\nFinal LightGBM rolling model logged successfully.")

2026/05/28 04:02:35 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm_rolling_optimized' does not exist. Creating a new experiment.
[I 2026-05-28 04:02:35,349] A new study created in memory with name: no-name-8c283892-2da7-4ee2-9df6-2de7ac2f40a3


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045582 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2937
[LightGBM] [Info] Number of data points in the train set: 138009, number of used features: 19
[LightGBM] [Info] Start training from score 138.705606


[I 2026-05-28 04:02:43,198] Trial 0 finished with value: 2.763871234418184 and parameters: {'n_estimators': 100, 'learning_rate': 0.017953465897151414, 'max_depth': 7, 'num_leaves': 50, 'min_child_samples': 35, 'subsample': 0.751022775525571, 'colsample_bytree': 0.9957210598106496, 'reg_alpha': 0.9142599869094749, 'reg_lambda': 0.0018489193523392724}. Best is trial 0 with value: 2.763871234418184.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024140 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2937
[LightGBM] [Info] Number of data points in the train set: 138009, number of used features: 19
[LightGBM] [Info] Start training from score 138.705606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

[I 2026-05-28 04:02:56,762] Trial 1 finished with value: 2.7034404545114534 and parameters: {'n_estimators': 150, 'learning_rate': 0.01192530858283102, 'max_depth': 7, 'num_leaves': 110, 'min_child_samples': 40, 'subsample': 0.9813762825037156, 'colsample_bytree': 0.9036392014130468, 'reg_alpha': 0.0003105316007655031, 'reg_lambda': 0.0001861470670335538}. Best is trial 1 with value: 2.7034404545114534.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.037702 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2937
[LightGBM] [Info] Number of data points in the train set: 138009, number of used features: 19
[LightGBM] [Info] Start training from score 138.705606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

[I 2026-05-28 04:17:42,103] Trial 49 finished with value: 0.6213062871043522 and parameters: {'n_estimators': 100, 'learning_rate': 0.038218382040081614, 'max_depth': 6, 'num_leaves': 60, 'min_child_samples': 40, 'subsample': 0.9749575148404123, 'colsample_bytree': 0.7947632816909108, 'reg_alpha': 0.00047128079695764606, 'reg_lambda': 0.002597012664225232}. Best is trial 32 with value: 0.3374257695027899.


Best Params:
{'n_estimators': 450, 'learning_rate': 0.06135835528376651, 'max_depth': 4, 'num_leaves': 90, 'min_child_samples': 10, 'subsample': 0.9669927599410787, 'colsample_bytree': 0.7429281810997791, 'reg_alpha': 0.00164067221857848, 'reg_lambda': 0.000556629777446131}

Best Validation MAPE:
0.3374257695027899
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.155028 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2942
[LightGBM] [Info] Number of data points in the train set: 172512, number of used features: 19
[LightGBM] [Info] Start training from score 124.424828
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

2026/05/28 04:17:52 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Final LightGBM rolling model logged successfully.


In [69]:
import mlflow
import mlflow.lightgbm
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ===================================================
# CREATE TIME FEATURES FOR TRAIN DATA
# ===================================================

df = train_df.copy()

# ensure datetime index
df.index = pd.to_datetime(df.index)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = df.index.day_name()

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# EWMA FEATURES
# ---------------------------------------------------

df["ewma_24"] = (
    df["total_pickups"]
    .shift(1)
    .ewm(
        span=24,
        adjust=False
    )
    .mean()
)

df["ewma_168"] = (
    df["total_pickups"]
    .shift(1)
    .ewm(
        span=168,
        adjust=False
    )
    .mean()
)

# ---------------------------------------------------
# OPTIONAL VOLATILITY FEATURE
# ---------------------------------------------------

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# CREATE SAME FEATURES FOR TEST DATA
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL ENCODING
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# EWMA FEATURES
# ---------------------------------------------------

test_processed["ewma_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .ewm(
        span=24,
        adjust=False
    )
    .mean()
)

test_processed["ewma_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .ewm(
        span=168,
        adjust=False
    )
    .mean()
)

# ---------------------------------------------------
# OPTIONAL VOLATILITY FEATURE
# ---------------------------------------------------

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(window=24)
    .std()
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = test_processed.dropna()

# ===================================================
# FEATURES AND TARGET
# ===================================================

# Jan + Feb
X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

# March
X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed["total_pickups"]

# ===================================================
# CATEGORICAL FEATURES
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ===================================================
# TRAIN VALIDATION SPLIT
# ===================================================

split_idx = int(
    len(X_full) * 0.8
)

X_train = X_full.iloc[:split_idx]

y_train = y_full.iloc[:split_idx]

X_val = X_full.iloc[split_idx:]

y_val = y_full.iloc[split_idx:]

# ===================================================
# LIGHTGBM REQUIRES NUMERIC CATEGORIES
# ===================================================

X_train_lgb = X_train.copy()

X_val_lgb = X_val.copy()

X_test_lgb = X_test.copy()

for col in categorical_cols:

    X_train_lgb[col] = (
        X_train_lgb[col]
        .cat.codes
    )

    X_val_lgb[col] = (
        X_val_lgb[col]
        .cat.codes
    )

    X_test_lgb[col] = (
        X_test_lgb[col]
        .cat.codes
    )

# ===================================================
# MLFLOW EXPERIMENT
# ===================================================

mlflow.set_experiment(
    "lightgbm_ewma_optimized"
)

# ===================================================
# OBJECTIVE FUNCTION
# ===================================================

def objective(trial):

    with mlflow.start_run(nested=True):

        model = LGBMRegressor(

            # ------------------------------------------------
            # IMPROVED SEARCH SPACE
            # ------------------------------------------------

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                500,
                step=50
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                1e-2,
                1e-1,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                8
            ),

            num_leaves=trial.suggest_int(
                "num_leaves",
                20,
                150,
                step=10
            ),

            min_child_samples=trial.suggest_int(
                "min_child_samples",
                10,
                50,
                step=5
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                1e-4,
                5,
                log=True
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                1e-4,
                5,
                log=True
            ),

            objective="regression",

            random_state=42,

            n_jobs=-1
        )

        # ---------------------------------------------------
        # TRAIN
        # ---------------------------------------------------

        model.fit(

            X_train_lgb,
            y_train,

            eval_set=[(X_val_lgb, y_val)],

            eval_metric="mae"
        )

        # ---------------------------------------------------
        # VALIDATION PREDICTIONS
        # ---------------------------------------------------

        y_pred = model.predict(
            X_val_lgb
        )

        # ---------------------------------------------------
        # METRICS
        # ---------------------------------------------------

        mape = mean_absolute_percentage_error(
            y_val,
            y_pred
        )

        mae = mean_absolute_error(
            y_val,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        r2 = r2_score(
            y_val,
            y_pred
        )

        # ---------------------------------------------------
        # LOGGING
        # ---------------------------------------------------

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "MAPE",
            mape
        )

        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return mape

# ===================================================
# OPTUNA TUNING
# ===================================================

with mlflow.start_run(

    run_name="lightgbm_ewma_tuning"

):

    study = optuna.create_study(
        direction="minimize"
    )

    study.optimize(

        objective,

        n_trials=50,

        n_jobs=1
    )

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "best_validation_MAPE",
        study.best_value
    )

print("Best Params:")
print(study.best_params)

print("\nBest Validation MAPE:")
print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

best_params = study.best_params

final_model = LGBMRegressor(

    **best_params,

    objective="regression",

    random_state=42,

    n_jobs=-1
)

# ===================================================
# COMBINE TRAIN + VALIDATION
# ===================================================

X_final_train = pd.concat([
    X_train_lgb,
    X_val_lgb
])

y_final_train = pd.concat([
    y_train,
    y_val
])

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_final_train,
    y_final_train
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test_lgb
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL TEST RESULTS")

print("MAPE:", final_mape)

print("MAE:", final_mae)

print("RMSE:", final_rmse)

print("R2:", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name="best_lightgbm_ewma_model"

):

    mlflow.log_params(
        best_params
    )

    mlflow.log_metric(
        "FINAL_MAPE",
        final_mape
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.lightgbm.log_model(
        lgb_model=final_model,
        name="model"
    )

print("\nFinal LightGBM EWMA model logged successfully.")

2026/05/28 03:49:32 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm_ewma_optimized' does not exist. Creating a new experiment.
[I 2026-05-28 03:49:32,152] A new study created in memory with name: no-name-c171be9c-7263-4b00-8032-398fa84d293d


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002644 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2682
[LightGBM] [Info] Number of data points in the train set: 138009, number of used features: 18
[LightGBM] [Info] Start training from score 138.705606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

[I 2026-05-28 03:49:34,045] Trial 0 finished with value: 0.8720707938224521 and parameters: {'n_estimators': 150, 'learning_rate': 0.021448383007989475, 'max_depth': 6, 'num_leaves': 60, 'min_child_samples': 10, 'subsample': 0.8978205810225773, 'colsample_bytree': 0.7740551637041865, 'reg_alpha': 0.12466313438268875, 'reg_lambda': 0.26537423109689107}. Best is trial 0 with value: 0.8720707938224521.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011814 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2682
[LightGBM] [Info] Number of data points in the train set: 138009, number of used features: 18
[LightGBM] [Info] Start training from score 138.705606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

[I 2026-05-28 03:49:35,075] Trial 1 finished with value: 1.4374717222405158 and parameters: {'n_estimators': 250, 'learning_rate': 0.01132233154934213, 'max_depth': 3, 'num_leaves': 30, 'min_child_samples': 20, 'subsample': 0.9249747053062107, 'colsample_bytree': 0.9278690959034241, 'reg_alpha': 0.06141223404675975, 'reg_lambda': 2.8579998238871305}. Best is trial 0 with value: 0.8720707938224521.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020092 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2682
[LightGBM] [Info] Number of data points in the train set: 138009, number of used features: 18
[LightGBM] [Info] Start training from score 138.705606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

[I 2026-05-28 04:00:57,943] Trial 49 finished with value: 0.36827019516995724 and parameters: {'n_estimators': 400, 'learning_rate': 0.0935525806896575, 'max_depth': 4, 'num_leaves': 70, 'min_child_samples': 15, 'subsample': 0.9467359161409613, 'colsample_bytree': 0.7333087367901773, 'reg_alpha': 0.004472888948590591, 'reg_lambda': 0.018069991548293286}. Best is trial 45 with value: 0.33538168808178026.


Best Params:
{'n_estimators': 400, 'learning_rate': 0.08720296683018487, 'max_depth': 5, 'num_leaves': 70, 'min_child_samples': 20, 'subsample': 0.7164960534477214, 'colsample_bytree': 0.7760546634224972, 'reg_alpha': 0.0003470225971712951, 'reg_lambda': 0.0036108313208778153}

Best Validation MAPE:
0.33538168808178026
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.181119 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2687
[LightGBM] [Info] Number of data points in the train set: 172512, number of used features: 18
[LightGBM] [Info] Start training from score 124.424828
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

2026/05/28 04:02:01 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Final LightGBM EWMA model logged successfully.
